<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T06-L01-retrieval-from-scratch/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T06-L01-retrieval-from-scratch/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T06-L01-retrieval-from-scratch/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T06-L01-retrieval-from-scratch/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T06-L01 · Retrieval from scratch, and a measurement you can trust

**You will build:** a lexical retriever (BM25), a dense retriever with no external model
(TF-IDF folded through a truncated SVD), a fusion of the two, and — first — the evaluation
harness that says whether any of it is any good.

**Time:** ~70 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no download ·
**Prerequisites:** T00-L01-the-8gb-track.

Most retrieval tutorials call an embedding API and eyeball three results. This lesson never
calls one: every score is computed by code in this file, on a corpus generated in this file,
so nothing here can be a demo you take on faith.

By the end you will be able to:

1. Implement BM25 from its primary-literature formula, and measure what its `+0.5`
   smoothing does for a term that sits in every document.
2. Implement a dense retriever with no external model — TF-IDF folded through a truncated
   SVD — with cosine scoring and normalisation done correctly at both ends.
3. Implement reciprocal rank fusion and measure what its constant does to the blend.
4. Implement recall@k, MRR and nDCG@k, and bootstrap confidence intervals over queries.
5. Measure, on a corpus built so that identifiers and paraphrases favour different
   retrievers, when hybrid's advantage over each pure method is real and when it is not.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import math
import re
import statistics
import sys
import traceback
from typing import Any, Callable, Mapping, NamedTuple, Sequence

import numpy as np

print("python", sys.version.split()[0], "· numpy", np.__version__, "· platform", sys.platform)

SEED = 20260923          # this lesson's one fixed seed: the corpus, the queries and every
                          # bootstrap resample all derive from it, so re-running prints the
                          # same numbers on any machine.
EVAL_K = 5                # the cutoff used for recall@k and nDCG@k throughout this lesson.
N_BOOT = 2000             # bootstrap resamples for every confidence interval below.

_FAILED_CHECKS: list[str] = []
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("tokenize", "build_index"),
    "exercise 2": ("bm25_idf", "bm25_score"),
    "exercise 3": ("bm25_rank",),
    "exercise 4": ("tfidf_matrix",),
    "exercise 5": ("truncated_svd",),
    "exercise 6": ("build_dense_index", "embed_query", "dense_rank"),
    "exercise 7": ("reciprocal_rank_fusion",),
    "exercise 8": ("recall_at_k", "reciprocal_rank", "ndcg_at_k"),
    "exercise 9": ("bootstrap_ci", "paired_bootstrap_diff"),
}


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (bm25_rank)"; several -> "exercises 3, 6 and 7"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")

## 1. The corpus: ten subjects, two ways to ask about each one (given)

`build_corpus()` writes short "case notes" about ten unrelated subjects (hydraulics,
networking, finance-ops, ...), with four kinds of document per subject:

- one **target** document, carrying a unique case code (`case-4800`, ...) plus three of the
  subject's own words;
- two **sibling** documents, carrying a *different* case code but the *same* three words —
  a near-duplicate that only a code, not a topic, can tell apart from the target;
- two **bridge** documents, carrying the subject's own words *and* a second, disjoint set of
  plain-English words for the same subject — the only place the two vocabularies meet;
- two **decoy** documents, carrying three of the subject's own words between generic ones:
  on topic, but never the answer.

Twenty-five unrelated "noise" documents (office small-talk) pad the corpus out. Every
document, deliberately, also contains the word "note" — you will meet why in exercise 2.
Each subject's documents also carry two filler words drawn at random, and a sibling never
repeats its target's pair, so no two documents differ ONLY by their code. If two did, the
dense retriever would score them exactly alike, and which one ranked first would be settled
by rounding in the last digit — which differs from one machine to the next.

Two queries per subject: an **exact** query (just the case code) and a **paraphrase** query
(two of the plain-English words, chosen so they never once appear in the target document —
only in its bridge documents). The target document is the one graded answer for both.

In [ ]:
TOPICS = [
    dict(name="hydraulics", vocab=["pump", "valve", "piston", "cylinder", "manifold"],
         synonym=["leak", "hose", "gauge", "washer", "fitting"]),
    dict(name="networking", vocab=["router", "switch", "packet", "firewall", "subnet"],
         synonym=["connection", "network", "latency", "outage", "gateway"]),
    dict(name="finance-ops", vocab=["invoice", "ledger", "remittance", "creditor", "payable"],
         synonym=["bill", "account", "payment", "supplier", "owed"]),
    dict(name="ceramics", vocab=["kiln", "glaze", "bisque", "clay", "slip"],
         synonym=["oven", "coating", "unfired", "mud", "mixture"]),
    dict(name="spacecraft", vocab=["orbit", "perigee", "thruster", "telemetry", "payload"],
         synonym=["loop", "closest", "engine", "signal", "cargo"]),
    dict(name="gardening", vocab=["compost", "mulch", "trellis", "loam", "seedling"],
         synonym=["rot", "cover", "stake", "soil", "sprout"]),
    dict(name="cryptography", vocab=["cipher", "nonce", "keypair", "checksum", "handshake"],
         synonym=["code", "random", "keys", "hash", "greeting"]),
    dict(name="hydrogeology", vocab=["aquifer", "borehole", "watershed", "sediment", "runoff"],
         synonym=["groundwater", "well", "catchment", "silt", "drainage"]),
    dict(name="construction", vocab=["scaffold", "rebar", "formwork", "truss", "joist"],
         synonym=["frame", "bar", "mould", "beam", "support"]),
    dict(name="mycology", vocab=["mycelium", "spore", "substrate", "hyphae", "inoculate"],
         synonym=["fungus", "seed", "medium", "thread", "plant"]),
]
FILLER_POOL = ["team", "today", "please", "status", "system", "regarding"]
GENERIC_NOISE_WORDS = [
    "meeting", "calendar", "printer", "coffee", "parking", "elevator", "cafeteria", "badge",
    "laptop", "monitor", "keyboard", "mouse", "chair", "desk", "window", "hallway",
    "stairwell", "lobby", "reception", "mailroom", "supply", "closet", "whiteboard",
    "projector", "conference", "schedule", "holiday", "vacation", "weather", "traffic",
]
N_SIBLINGS = 2
N_NOISE = 25

# Every topic's ten words are unique to it, and vocab and synonym words never overlap: this
# is what lets a truncated SVD find one subject per latent axis in section 6. Checked once,
# here, rather than trusted.
_all_vocab = [w for t in TOPICS for w in t["vocab"]]
_all_synonym = [w for t in TOPICS for w in t["synonym"]]
assert len(_all_vocab) == len(set(_all_vocab)), "two topics share a vocab word"
assert len(_all_synonym) == len(set(_all_synonym)), "two topics share a synonym word"
assert not (set(_all_vocab) & set(_all_synonym)), "a vocab word is also a synonym word"
assert not (set(_all_vocab) & set(GENERIC_NOISE_WORDS))
assert not (set(_all_synonym) & set(GENERIC_NOISE_WORDS))


def build_corpus(seed: int = SEED) -> tuple[list[str], list[tuple[int, str]]]:
    """Generate the ten-subject corpus described above from a numpy Generator seeded `seed`.

    Returns `(docs, doc_meta)`: `docs[i]` is document `i`'s text; `doc_meta[i]` is
    `(topic_index, role)` with `role` one of "target", "sibling0", "sibling1", "bridgeA",
    "bridgeB", "decoy1", "decoy2", "noise" (topic_index is -1 for noise documents).
    """
    rng = np.random.default_rng(seed)
    docs: list[str] = []
    doc_meta: list[tuple[int, str]] = []

    def pick_filler() -> tuple[str, str]:
        a, b = rng.choice(FILLER_POOL, size=2, replace=False).tolist()
        return a, b

    for ti, topic in enumerate(TOPICS):
        identifier = f"case-{4800 + ti * 37}"
        v, s = topic["vocab"], topic["synonym"]
        f1, f2 = pick_filler()
        target_filler = {f1, f2}
        docs.append(f"{identifier} note: {v[0]} {v[1]} {v[2]} status update, {f1} {f2} "
                    f"logged today")
        doc_meta.append((ti, "target"))
        for si in range(N_SIBLINGS):
            sib_id = f"case-{9000 + ti * 37 + si}"
            f1, f2 = pick_filler()
            while {f1, f2} == target_filler:   # never the target's own pair: see section 1
                f1, f2 = pick_filler()
            docs.append(f"{sib_id} note: {v[0]} {v[1]} {v[2]} status update, {f1} {f2} "
                        f"logged today")
            doc_meta.append((ti, f"sibling{si}"))
        f1, f2 = pick_filler()
        docs.append(f"note: {v[0]} {s[0]} {v[1]} {s[1]} {v[2]} {s[2]} {f1} {f2}")
        doc_meta.append((ti, "bridgeA"))
        f1, f2 = pick_filler()
        docs.append(f"note: {v[2]} {s[2]} {v[3]} {s[3]} {v[4]} {s[4]} {f1} {f2}")
        doc_meta.append((ti, "bridgeB"))
        f1, f2 = pick_filler()
        docs.append(f"note: general {v[1]} {v[2]} {v[3]} reference {f1} {f2}")
        doc_meta.append((ti, "decoy1"))
        f1, f2 = pick_filler()
        docs.append(f"note: general {v[2]} {v[3]} {v[4]} reference {f1} {f2}")
        doc_meta.append((ti, "decoy2"))
    for _ in range(N_NOISE):
        words = rng.choice(GENERIC_NOISE_WORDS, size=5, replace=False).tolist()
        docs.append("note: " + " ".join(words))
        doc_meta.append((-1, "noise"))

    order = rng.permutation(len(docs))          # shuffle so topic order is not corpus order
    docs = [docs[i] for i in order]
    doc_meta = [doc_meta[i] for i in order]
    return docs, doc_meta


class Query(NamedTuple):
    """One evaluation query and the judgements it is graded against.

    `relevant_ids` (a frozenset) is the primary answer, used by recall@k and MRR. `relevance`
    is the fuller graded judgement used by nDCG@k: the primary answer at grade 2, plus this
    subject's two bridge documents at grade 1 — genuinely on-subject, but not what an exact
    case lookup is really asking for, and not enough on their own for recall or MRR to count.
    """

    qid: str
    tokens: list[str]
    kind: str                      # "exact" | "paraphrase"
    relevant_ids: frozenset[int]
    relevance: dict[int, int]


def build_queries(doc_meta: Sequence[tuple[int, str]]) -> list[Query]:
    """Build the two queries per subject (exact, paraphrase) against `doc_meta` from build_corpus.

    The paraphrase query is each subject's first two synonym words — words that, by
    construction, never appear in the target document and appear only in its bridge documents.
    """
    by_topic: dict[int, dict[str, int]] = {}
    for doc_id, (ti, role) in enumerate(doc_meta):
        if ti == -1:
            continue
        by_topic.setdefault(ti, {})[role] = doc_id

    queries: list[Query] = []
    for ti, topic in enumerate(TOPICS):
        ids = by_topic[ti]
        target_id = ids["target"]
        relevance = {target_id: 2, ids["bridgeA"]: 1, ids["bridgeB"]: 1}
        identifier = f"case-{4800 + ti * 37}"
        queries.append(Query(f"T{ti}-exact", tokenize(identifier), "exact",
                              frozenset({target_id}), dict(relevance)))
        queries.append(Query(f"T{ti}-paraphrase", list(topic["synonym"][:2]), "paraphrase",
                              frozenset({target_id}), dict(relevance)))
    return queries

## 2. Exercise 1 — the tokeniser and the index

Every retriever below reads the corpus through the same two functions. `tokenize` turns raw
text into tokens; `build_index` counts them once so nothing downstream re-scans the corpus.

<details><summary>💡 Hint 1 — what to think about</summary>

A token is a run of letters and digits, optionally joined by single hyphens, so a code like
`"case-4800"` survives as one token rather than splitting at the hyphen. For the index, ask
what `df` is really counting: does a word appearing three times in one document add 1 or 3
to that word's document frequency?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

`tokenize`: lower-case the text, then find every run matching letters/digits/hyphens with
one regular expression. `build_index`: for each document, count its own tokens into a fresh
dict; add that document's LENGTH (the sum of its own counts) to `doc_lengths`; then, once per
DISTINCT term the document contains (its counts dict's own keys), add one to that term's
entry in the corpus-wide `df`. `avgdl` is the mean of `doc_lengths`.
</details>

In [ ]:
_TOKEN_RE = re.compile(r"[a-z0-9]+(?:-[a-z0-9]+)*")


def tokenize(text: str) -> list[str]:
    """Lower-case `text` and split it into tokens.

    A token is a run of letters and digits, optionally joined by single hyphens — so
    `"case-4800"` is one token, not three.

    Example:
        >>> tokenize("Case-4800 note: PUMP failure")
        ['case-4800', 'note', 'pump', 'failure']
    """
    # YOUR CODE HERE
    raise NotImplementedError


class Index(NamedTuple):
    """A tokenised corpus, indexed once for every retriever built on it."""

    doc_term_counts: list[dict[str, int]]  # doc_term_counts[d][term] = count of term in doc d
    doc_lengths: list[int]                 # token count of each document
    avgdl: float                           # mean of doc_lengths over the whole corpus
    df: dict[str, int]                     # document frequency: how many docs contain term
    n_docs: int


def build_index(docs: Sequence[str]) -> Index:
    """Tokenise every document in `docs` and build the counts every retriever below reads.

    Requirements, each of which is graded:
      * `doc_term_counts[d]` is a `{term: count}` dict of document `d`'s OWN tokens.
      * `doc_lengths[d]` is document `d`'s total token count (the sum of its own counts).
      * `df[term]` counts how many DOCUMENTS contain `term` at least once — not how many times
        the term occurs across the corpus. A term appearing 3 times in one document still
        contributes only 1 to that term's `df`.
      * `avgdl` is the mean of `doc_lengths` over the whole corpus (0.0 if there are no docs).

    Example:
        >>> idx = build_index(["a b b", "b c"])
        >>> idx.doc_term_counts[0]
        {'a': 1, 'b': 2}
        >>> idx.df
        {'a': 1, 'b': 2, 'c': 1}
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_index() -> None:
    assert tokenize("Case-4800 note!") == ["case-4800", "note"], (
        "tokenize must lower-case and keep a hyphenated code as ONE token, dropping "
        "punctuation that isn't part of one"
    )
    idx = build_index(["pump valve pump", "valve gauge"])
    assert idx.n_docs == 2, f"n_docs should be 2, got {idx.n_docs}"
    assert idx.doc_term_counts[0] == {"pump": 2, "valve": 1}, (
        f"doc 0 should count pump=2, valve=1 — got {idx.doc_term_counts[0]}"
    )
    assert idx.doc_lengths == [3, 2], f"doc_lengths should be [3, 2], got {idx.doc_lengths}"
    assert idx.avgdl == 2.5, f"avgdl should be (3+2)/2 = 2.5, got {idx.avgdl}"
    assert idx.df == {"pump": 1, "valve": 2, "gauge": 1}, (
        f"df counts DOCUMENTS containing a term, not occurrences — got {idx.df}"
    )
    print("exercise 1 looks right — tokenize() and build_index() agree with a hand-counted "
          "two-document corpus")

In [ ]:
_try("exercise 1", _check_index)

In [ ]:
def _show_corpus_sample() -> None:
    docs, doc_meta = build_corpus()
    index = build_index(docs)
    sample = next(i for i, (_, role) in enumerate(doc_meta) if role == "target")
    print(f"corpus: {index.n_docs} documents, {len(index.df)} distinct tokens, "
          f"avgdl={index.avgdl:.1f} tokens/doc")
    print(f"\na target document: {docs[sample]!r}")
    print(f"its tokens: {tokenize(docs[sample])}")


_try("corpus sample", _show_corpus_sample, needs=("exercise 1",))

## 3. Exercise 2 — BM25's weight for one term: `bm25_idf` and `bm25_score`

BM25's document score is a sum, over the query's terms, of an inverse-document-frequency
weight times a saturating term-frequency weight. Both are in `claims.yaml`, sourced from
Robertson and Zaragoza (2009). The IDF half:

$$\mathrm{idf}(t) = \ln\left(\frac{N - n_t + 0.5}{n_t + 0.5}\right)$$

where $N$ is the corpus size and $n_t$ the number of documents containing $t$. The full
score, with $\mathrm{tf}$ the term's count in the document, $dl$ that document's length and
$\mathrm{avgdl}$ the corpus average:

$$\mathrm{score}(q, d) = \sum_{t \in q} \mathrm{qtf}(t) \cdot \mathrm{idf}(t) \cdot
\frac{\mathrm{tf}(t, d)}{k_1\left((1-b) + b \frac{dl}{\mathrm{avgdl}}\right) + \mathrm{tf}(t, d)}$$

`qtf(t)` is how many times `t` itself appears in the QUERY — repeating a query word is not a
no-op. `k1` (saturation) and `b` (length normalisation) default to 1.2 and 0.75.

<details><summary>💡 Hint 1 — what to think about</summary>

Try `bm25_idf` on a term that sits in EVERY document of a tiny corpus by hand, on paper,
before running anything: what is `N - n_t` when `n_t == N`? Is the fraction bigger or
smaller than 1, and what does `ln` of a fraction smaller than 1 come out as? Now ask what a
formula WITHOUT the two `+0.5` constants would compute for that same case.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

`bm25_idf`: take the term's document frequency from the index (zero for a word it has never
seen) and the number of documents, and evaluate the formula above with the natural log —
nothing more. `bm25_score`: first count how often each word occurs in the QUERY, so a
repeated word is not lost. Then, for each distinct query word, look up how often it occurs
in this document; skip the word if that is zero, otherwise add the score formula's term for
it, multiplied by its query count, to a running total.
</details>

In [ ]:
def bm25_idf(term: str, index: Index) -> float:
    """BM25's inverse-document-frequency weight for `term` against the whole corpus in `index`.

    Requirements, each of which is graded:
      * the formula above, read from `index.df` and `index.n_docs` — no clamping, no flooring
        at zero.
      * when `term` is in every document (`n_t == N`), this must return a FINITE, NEGATIVE
        number, never `-inf`, `nan`, or a raised exception. The two `+0.5` constants exist to
        make exactly that case safe.

    Example:
        >>> idx = build_index(["a b", "a c", "a d"])   # 'a' is in all 3 documents
        >>> bm25_idf("a", idx) < 0.0
        True
        >>> bm25_idf("b", idx) > bm25_idf("a", idx)     # a rarer term scores higher
        True
    """
    # YOUR CODE HERE
    raise NotImplementedError


def bm25_score(query_tokens: Sequence[str], doc_id: int, index: Index,
               k1: float = 1.2, b: float = 0.75) -> float:
    """BM25's score for document `doc_id` against `query_tokens`, using the formula above.

    Requirements, each of which is graded:
      * a query term that never repeats contributes `idf(t) * tf / (...)` as usual; a query
        term that repeats `qtf` times contributes `qtf` times that much — count the query's
        own term frequencies, do not deduplicate.
      * a query term absent from the document (`tf == 0`) contributes exactly `0.0` and should
        not even need `bm25_idf` called for it.
      * `b` controls length normalisation: at `b = 0.0` it must have no effect at all.
      * the value is the formula above. Some published versions also multiply each term by
        `k1 + 1`; Robertson and Zaragoza note that factor is the same for every term and
        leaves the ranking unchanged (see `claims.yaml`), so either form is accepted.

    Example:
        >>> idx = build_index(["pump valve", "gauge gauge"])
        >>> bm25_score(["pump"], 0, idx) > 0.0
        True
        >>> bm25_score(["pump"], 1, idx)   # 'pump' never appears in document 1
        0.0
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_bm25_core() -> None:
    idx = build_index(["case-1 pump valve", "case-2 pump gauge", "case-3 pump manifold"])
    every_doc_idf = bm25_idf("pump", idx)      # 'pump' is in every one of the 3 documents
    assert every_doc_idf < 0.0 and math.isfinite(every_doc_idf), (
        f"bm25_idf('pump', ...) should be a finite NEGATIVE number when the term is in every "
        f"document, got {every_doc_idf} — check the two '+ 0.5' terms in the formula"
    )
    rare_idf = bm25_idf("case-1", idx)          # 'case-1' is in exactly one document
    assert rare_idf > every_doc_idf, (
        "a term in only one document must score a HIGHER idf than one in every document"
    )
    absent_idf = bm25_idf("nope", idx)
    assert absent_idf > rare_idf, "a term in NO document should score the highest idf of all"

    score_present = bm25_score(["pump"], 0, idx)
    score_absent = bm25_score(["nope"], 0, idx)
    assert score_absent == 0.0, "a query term absent from the document must contribute exactly 0"
    assert score_present != 0.0, "a query term present in the document must contribute something"

    once = bm25_score(["case-1"], 0, idx)
    twice = bm25_score(["case-1", "case-1"], 0, idx)
    assert abs(twice - 2 * once) < 1e-9, (
        f"repeating a query term should double its contribution (qtf=2), got {once:.4f} then "
        f"{twice:.4f} — count each query term's own frequency (qtf) and multiply by it"
    )
    print("exercise 2 looks right — bm25_idf handles the all-documents edge case, and "
          "bm25_score weights a repeated query term by how often it repeats")

In [ ]:
_try("exercise 2", _check_bm25_core)

In [ ]:
def _show_idf_edge_case() -> None:
    docs, doc_meta = build_corpus()
    index = build_index(docs)
    note_idf = bm25_idf("note", index)
    print(f"'note' appears in {index.df.get('note')} of {index.n_docs} documents "
          f"(by construction: every document).")
    print(f"bm25_idf('note', index) = {note_idf:+.3f}")
    rare_term = TOPICS[0]["vocab"][0]
    print(f"bm25_idf({rare_term!r}, index) = {bm25_idf(rare_term, index):+.3f}  "
          f"(df={index.df.get(rare_term)})")
    top = max(bm25_idf(term, index) for term in index.df)
    print(f"the largest weight any term in this corpus gets: {top:+.3f}")
    print(f"\n'note' is weighted {abs(note_idf) / top:.2f} times that size, with a minus sign. A term")
    print("in every document is not ignored: section 4 shows what it does to a query.")


_try("idf edge case demo", _show_idf_edge_case, needs=("exercise 2",))

## 4. Exercise 3 — `bm25_rank`: ranking, and removing the zeros

Robertson and Zaragoza call the next step "removing the zeros": a real search engine, built
on an inverted index, only ever looks up documents that share at least one term with the
query — a document with none is never scored at all, not scored zero. `bm25_rank` must do
the same: a real system cannot rank what it never retrieved.

<details><summary>💡 Hint — the approach, in words</summary>

Score every document with `bm25_score`, keep only the ones whose score is not exactly
`0.0`, then sort what remains by score descending — and, for documents that tie exactly on
score, by document id ascending, so the ranking is the same every time it is computed.
</details>

In [ ]:
def bm25_rank(query_tokens: Sequence[str], index: Index,
              k1: float = 1.2, b: float = 0.75) -> list[tuple[int, float]]:
    """Rank every document in `index` against `query_tokens` by BM25 score, best first.

    Requirements, each of which is graded:
      * a document scoring exactly `0.0` — which is what a document sharing no term with the
        query scores — is REMOVED from the ranking, not included at score 0.0. A document
        scoring below zero is not a zero, and stays.
      * the remaining documents are sorted by score descending; ties break by document id
        ascending.

    Example:
        >>> idx = build_index(["pump valve", "gauge only"])
        >>> bm25_rank(["pump"], idx)
        [(0, ...)]
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_bm25_rank() -> None:
    idx = build_index(["case-1 pump valve", "case-2 gauge only", "unrelated noise words"])
    ranked = bm25_rank(["pump"], idx)
    ranked_ids = [d for d, _ in ranked]
    assert ranked_ids == [0], (
        f"only document 0 shares a term with the query 'pump' — expected ranked ids [0], "
        f"got {ranked_ids}. A document with no shared term must be DROPPED, not scored 0"
    )
    assert bm25_rank(["completely", "absent", "terms"], idx) == [], (
        "a query that shares no term with any document must rank NO documents at all"
    )
    scores = [s for _, s in bm25_rank(["pump", "valve"], idx)]
    assert scores == sorted(scores, reverse=True), "results must be sorted best score first"
    print("exercise 3 looks right — bm25_rank drops documents with zero shared terms instead "
          "of ranking every document in the corpus")

In [ ]:
_try("exercise 3", _check_bm25_rank)

In [ ]:
def _show_bm25_on_both_query_kinds() -> None:
    docs, doc_meta = build_corpus()
    index = build_index(docs)
    queries = build_queries(doc_meta)
    exact_q = next(q for q in queries if q.kind == "exact")
    para_q = next(q for q in queries if q.kind == "paraphrase")
    for q in (exact_q, para_q):
        ranked = bm25_rank(q.tokens, index)
        hit = ranked and ranked[0][0] in q.relevant_ids
        print(f"{q.kind:10s} query {q.tokens} -> {len(ranked)} documents share a term; "
              f"top result is the target: {bool(hit)}")
    word = TOPICS[0]["vocab"][0]
    plain = bm25_rank([word], index)
    with_note = bm25_rank([word, "note"], index)
    below = sum(score < 0.0 for _, score in with_note)
    print(f"\nquery [{word!r}]: {len(plain)} of {index.n_docs} documents ranked, best score "
          f"{plain[0][1]:+.3f}")
    print(f"query [{word!r}, 'note']: {len(with_note)} of {index.n_docs} documents ranked, best "
          f"score {with_note[0][1]:+.3f}, {below} scoring below zero")
    print("\nadding a word that every document contains subtracts from every document's score,")
    print("and a document scoring below zero is not a zero: nothing is removed for it.")


_try("bm25 on exact vs paraphrase", _show_bm25_on_both_query_kinds, needs=("exercise 3",))

## 5. Exercise 4 — a dense document as a vector: `tfidf_matrix`

BM25 can only ever match a literal token. The rest of this lesson builds a retriever with no
such restriction and no external model: turn every document into a numeric vector (this
exercise), fold that matrix through a truncated SVD (exercise 5), then rank by the angle
between vectors (exercise 6).

`tfidf_idf` below (not graded — read it, you will reuse the pattern) is a smoothed,
always-non-negative idf, deliberately different from `bm25_idf`: a vector space has no
"removing the zeros" step to protect it from a large negative weight swamping a sparse row.

<details><summary>💡 Hint — the approach, in words</summary>

Fix the column order first: the corpus's terms in alphabetical order, never the order a dict
or a set happens to iterate in. Start from a matrix of zeros. For each document, visit only
ITS OWN terms and write each one's weight — the sub-linear count times that column's idf —
into its column; every other column stays zero. Only when the whole matrix is built, divide
each row by its own length. A row with no tokens has length zero: divide it by one instead,
so it stays all zero rather than turning into `nan`.
</details>

In [ ]:
def tfidf_idf(term: str, index: Index) -> float:
    """A smoothed idf for the TF-IDF vector space: log(N / n_t) + 1, always >= 1.0.

    Given, not graded. Unlike `bm25_idf`, this can never go negative — a term in every
    document still gets weight 1.0, rather than actively penalising documents that contain it.
    """
    n_t = index.df.get(term, 0)
    if n_t == 0:
        return 0.0
    return math.log(index.n_docs / n_t) + 1.0


def tfidf_matrix(index: Index) -> tuple[np.ndarray, list[str], dict[str, int], np.ndarray]:
    """Build the corpus's TF-IDF matrix: one row per document, one column per distinct term.

    Returns `(M, vocab, vocab_pos, idf_weights)`: `vocab` is the corpus's terms in a fixed,
    reproducible order; `vocab_pos` maps a term to its column; `idf_weights[j]` is `tfidf_idf`
    for `vocab[j]`.

    Requirements, each of which is graded:
      * `vocab` is `sorted(index.df)` — alphabetical, reproducible.
      * a term's raw weight in its document's row is `(1 + log(tf)) * tfidf_idf(term)` — this
        sub-linear `1 + log(tf)` scaling is why a term occurring 4 times counts for less than
        4 separate single occurrences would.
      * every row of `M` is then divided by its own L2 norm, so it is unit length — EXCEPT a
        row that is already all zero (a document with no tokens at all, such as an empty or
        punctuation-only text), which must stay all zero rather than dividing zero by zero.

    Example:
        >>> idx = build_index(["pump valve", "gauge"])
        >>> M, vocab, vocab_pos, idf = tfidf_matrix(idx)
        >>> M.shape
        (2, 3)
        >>> abs(float(np.linalg.norm(M[0])) - 1.0) < 1e-9
        True
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_tfidf_matrix() -> None:
    idx = build_index(["pump pump valve", "valve gauge", "--- !!! ---"])   # doc 2: no tokens
    M, vocab, vocab_pos, idf_weights = tfidf_matrix(idx)
    assert vocab == sorted(vocab), "vocab must be in sorted (alphabetical) order, reproducibly"
    assert M.shape == (idx.n_docs, len(vocab)), f"M should be ({idx.n_docs}, {len(vocab)})"
    assert not np.isnan(M).any() and not M[2].any(), (
        "document 2 has no tokens at all, so its row must stay exactly zero — guard the "
        "division by a zero row norm instead of dividing 0 by 0, which gives nan"
    )
    for i in (0, 1):
        n = float(np.linalg.norm(M[i]))
        assert abs(n - 1.0) < 1e-9, (
            f"row {i} has norm {n:.4f} — divide every non-empty row by its own L2 norm"
        )
    assert idf_weights.min() >= 1.0, "tfidf_idf is always >= 1.0 — did you build idf_weights " \
        "from tfidf_idf, not bm25_idf?"
    raw_pump = (1.0 + math.log(2)) * tfidf_idf("pump", idx)
    raw_valve = tfidf_idf("valve", idx)
    expected = raw_pump / math.hypot(raw_pump, raw_valve)
    got = float(M[0, vocab_pos["pump"]])
    assert abs(got - expected) < 1e-9, (
        f"'pump' occurs twice in document 0; after normalising, its weight should be "
        f"{expected:.4f}, got {got:.4f} — weight a term by (1 + log(tf)) times its idf"
    )
    print("exercise 4 looks right — tfidf_matrix returns a matrix with unit-norm rows in a "
          "reproducible vocabulary order")

In [ ]:
_try("exercise 4", _check_tfidf_matrix)

## 6. Exercise 5 — dimensionality reduction: `truncated_svd`

`numpy.linalg.svd` factors any matrix `M` as `U @ diag(S) @ Vt`, with `S`'s entries sorted
largest first. Keeping only the first `k` columns of `U`, entries of `S` and rows of `Vt` is
a *truncated* SVD: the low-rank approximation that keeps the most shared structure in `M`
for the fewest dimensions — the mechanism Deerwester et al. (1990) call latent semantic
analysis (see `claims.yaml`).

<details><summary>💡 Hint — the approach, in words</summary>

numpy's SVD already returns the singular values largest first, with `U` and `Vt` in the same
order, so do not re-sort anything. Work out how many dimensions you can actually keep — never
more than there are singular values — then keep that many leading columns of `U`, leading
entries of `S` and leading rows of `Vt`.
</details>

In [ ]:
def truncated_svd(matrix: np.ndarray, k: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """The rank-`k` truncated SVD of `matrix`: `(U_k, S_k, Vt_k)`.

    Requirements, each of which is graded:
      * built from `np.linalg.svd(matrix, full_matrices=False)`, keeping the first `k` columns
        of `U`, entries of `S` and rows of `Vt` — `S`'s order must not be disturbed.
      * `k` is CLAMPED to the number of singular values actually available: asking for more
        dimensions than `matrix` has must not raise.

    Example:
        >>> U_k, S_k, Vt_k = truncated_svd(np.eye(5), k=2)
        >>> U_k.shape, S_k.shape, Vt_k.shape
        ((5, 2), (2,), (2, 5))
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_truncated_svd() -> None:
    M = np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0], [1.0, 1.0, 0.0]])
    U_k, S_k, Vt_k = truncated_svd(M, k=2)
    assert U_k.shape == (4, 2), f"U_k should be (4, 2), got {U_k.shape}"
    assert S_k.shape == (2,), f"S_k should be (2,), got {S_k.shape}"
    assert Vt_k.shape == (2, 3), f"Vt_k should be (2, 3), got {Vt_k.shape}"
    assert S_k[0] >= S_k[1] >= 0.0, "singular values must come back largest first"
    largest = np.linalg.svd(M, compute_uv=False)[:2]
    assert np.allclose(S_k, largest), (
        f"S_k should be the 2 LARGEST singular values {largest.round(4).tolist()}, got "
        f"{np.asarray(S_k).round(4).tolist()} — keep the FIRST k that np.linalg.svd returns"
    )
    reconstructed = (U_k * S_k) @ Vt_k
    assert np.linalg.norm(reconstructed - M) < np.linalg.norm(M), (
        "a rank-2 truncation of a rank-3 matrix should already recover most of it"
    )
    oversized = truncated_svd(M, k=100)
    assert oversized[1].shape[0] == min(M.shape), (
        "asking for more dimensions than the matrix has must be CLAMPED, not raise"
    )
    print("exercise 5 looks right — truncated_svd returns correctly-shaped, correctly-ordered "
          "factors and clamps an oversized k")

In [ ]:
_try("exercise 5", _check_truncated_svd)

## 7. Exercise 6 — the dense retriever: assembling, embedding, ranking

Three pieces: `build_dense_index` runs exercises 4 and 5 once and keeps what a query needs;
`embed_query` projects a NEW query into that same reduced space (never rebuilding the SVD);
`dense_rank` scores every document by cosine similarity to the query.

<details><summary>💡 Hint 1 — what to think about</summary>

A document is embedded once, in `build_dense_index`; a query is embedded LATER, possibly
many times, by `embed_query`. Whatever rule turns a document's TF-IDF row into a `k`-length
vector, `embed_query` must turn a query's own TF-IDF row into a `k`-length vector by the
EXACT SAME rule, or the two will not be comparable however correct each looks alone.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

`build_dense_index`: build the TF-IDF matrix (exercise 4), truncate it (exercise 5), project
every document row onto the kept directions as the docstring says, and give each projected
row unit length with the same zero-length guard as exercise 4. `embed_query`: build the
query's own TF-IDF row exactly as a document's row was built — counts first, the same
sub-linear weighting, the same idf — skipping any word the index has no column for; project
it onto the same kept directions; give it unit length unless its length is zero, in which
case hand it back untouched. `dense_rank`: an all-zero query scores every document zero;
otherwise one matrix-vector product gives every cosine at once. Order as in `bm25_rank`.
</details>

In [ ]:
class DenseIndex(NamedTuple):
    """A corpus indexed for dense retrieval: everything a query needs to be embedded and scored."""

    vocab_pos: dict[str, int]
    idf_weights: np.ndarray
    Vt_k: np.ndarray            # (k, vocab_size): the reduced-space basis, from exercise 5
    doc_embeddings: np.ndarray  # (n_docs, k), L2-normalised


def build_dense_index(index: Index, k: int) -> DenseIndex:
    """Build a `DenseIndex` for `index`, reduced to `k` latent dimensions.

    Requirements, each of which is graded:
      * documents are projected with `M @ Vt_k.T` — the same numbers as `U_k * S_k`, written
        this way because it is the rule a query, which has no row of `U`, can use too.
      * each embedded row is L2-normalised, with the same zero-norm guard as exercise 4.

    Example:
        >>> idx = build_index(["pump valve", "gauge only"])
        >>> di = build_dense_index(idx, k=2)
        >>> di.doc_embeddings.shape[0]
        2
    """
    # YOUR CODE HERE
    raise NotImplementedError


def embed_query(query_tokens: Sequence[str], dense_index: DenseIndex) -> np.ndarray:
    """Project `query_tokens` into `dense_index`'s reduced space as a unit-length vector.

    Requirements, each of which is graded:
      * an out-of-vocabulary token (absent from `dense_index.vocab_pos`) is simply skipped —
        it has no column to write into.
      * the projection rule is `dense_index.Vt_k @ vec`, matching `build_dense_index` exactly.
      * a query with no in-vocabulary tokens at all projects to the all-ZERO vector, returned
        as is — never divided by its own (zero) norm.

    Example:
        >>> idx = build_index(["pump valve", "gauge only"])
        >>> di = build_dense_index(idx, k=2)
        >>> v = embed_query(["pump"], di)
        >>> v.shape
        (2,)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def dense_rank(query_tokens: Sequence[str], dense_index: DenseIndex) -> list[tuple[int, float]]:
    """Rank every document in `dense_index` against `query_tokens` by cosine similarity.

    Requirements, each of which is graded:
      * because every document row and the query vector are unit length, cosine similarity IS
        the plain dot product — no separate normalisation step belongs here.
      * a zero-vector query (see `embed_query`) must score every document exactly `0.0`, never
        `nan`.
      * ties break by document id ascending, exactly as in `bm25_rank`.

    Example:
        >>> idx = build_index(["pump valve", "gauge only"])
        >>> di = build_dense_index(idx, k=2)
        >>> ranked = dense_rank(["pump"], di)
        >>> ranked[0][0]
        0
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_dense_retriever() -> None:
    # Two documents with no word in common and a third with no tokens at all: the matrix has
    # rank 2, so k=2 keeps all of it, and nothing below depends on which of two equal singular
    # values the SVD happens to list first.
    idx = build_index(["pump valve piston", "gauge hose fitting", "--- !!! ---"])
    di = build_dense_index(idx, k=2)
    assert di.doc_embeddings.shape == (idx.n_docs, 2), (
        f"doc_embeddings should be ({idx.n_docs}, 2), got {di.doc_embeddings.shape}"
    )
    row_norms = np.linalg.norm(di.doc_embeddings, axis=1)
    assert abs(row_norms[0] - 1.0) < 1e-6 and abs(row_norms[1] - 1.0) < 1e-6, (
        f"every embedded document row must be unit length, got norms {row_norms[:2]}"
    )
    assert row_norms[2] == 0.0, (
        "document 2 has no tokens, so its embedded row must stay exactly zero — guard the "
        "row-norm division as in exercise 4 instead of producing nan"
    )
    truncated = build_dense_index(build_index(["pump valve piston", "pump valve gauge",
                                               "pump valve hose", "compost mulch"]), k=2)
    cut_norms = np.linalg.norm(truncated.doc_embeddings, axis=1)
    assert np.allclose(cut_norms, 1.0, atol=1e-9), (
        f"after truncating to k=2 every embedded row must have unit length again, got "
        f"{np.round(cut_norms, 4).tolist()} — projecting onto fewer dimensions shortens a row, "
        "so normalise AFTER projecting"
    )
    empty_vec = embed_query(["zzqx", "wibble", "florp"], di)
    assert not np.any(empty_vec), (
        "a query built only from out-of-vocabulary words must embed to the all-zero vector"
    )
    empty_ranked = dense_rank(["zzqx", "wibble", "florp"], di)
    assert all(score == 0.0 for _, score in empty_ranked), (
        "ranking a zero-vector query must give every document a score of exactly 0.0, not NaN"
    )
    assert [d for d, _ in empty_ranked] == [0, 1, 2], (
        f"every document tied at 0.0, so the ranking must be in ascending id order, got "
        f"{[d for d, _ in empty_ranked]} — sort by (score descending, id ascending)"
    )
    on_topic = dense_rank(["pump"], di)
    assert on_topic[0][0] == 0, (
        f"document 0 shares 'pump' with the query and should rank first, got {on_topic[0]}"
    )
    print("exercise 6 looks right — the dense retriever embeds, normalises and ranks without "
          "producing a NaN for an out-of-vocabulary query")

In [ ]:
_try("exercise 6", _check_dense_retriever, needs=("exercise 4", "exercise 5"))

In [ ]:
DENSE_K = 15   # latent dimensions: about 1.5x the corpus's 10 real subjects


def _show_dense_on_both_query_kinds() -> None:
    docs, doc_meta = build_corpus()
    index = build_index(docs)
    dense_index = build_dense_index(index, DENSE_K)
    queries = build_queries(doc_meta)
    exact_q = next(q for q in queries if q.kind == "exact")
    para_q = next(q for q in queries if q.kind == "paraphrase")
    for q in (exact_q, para_q):
        ranked = dense_rank(q.tokens, dense_index)
        top_ids = [d for d, _ in ranked[:EVAL_K]]
        print(f"{q.kind:10s} query {q.tokens} -> target document in the top {EVAL_K}: "
              f"{any(d in q.relevant_ids for d in top_ids)}")
    print("\nthe paraphrase query shares NO token with its target document at all — dense")
    print("retrieval can only find it through the bridge documents' shared latent structure.")


_try("dense on exact vs paraphrase", _show_dense_on_both_query_kinds, needs=("exercise 6",))

## 8. Exercise 7 — combining rankings: `reciprocal_rank_fusion`

Reciprocal rank fusion (RRF) combines several ranked lists with no scores to calibrate and
no training data: a document's fused score is the sum, over every list that contains it, of
`1 / (k + rank)`. Cormack, Clarke and Büttcher (2009) fixed `k = 60` (see `claims.yaml`); a
document absent from a list simply contributes nothing from that list. The demo after the
check fuses this lesson's two retrievers and prints, query by query, where the target lands
at three values of `k`, so you can see what `k` does to this particular blend.

<details><summary>💡 Hint — the approach, in words</summary>

Keep one running total per document. Walk each input ranking from the top, counting ranks
from one, and add that position's share to the document's total — starting from nothing the
first time a document is seen. A document that appears in no ranking never gets a total at
all. Order the finished totals best first, and break an exact tie by the smaller id.
</details>

In [ ]:
def reciprocal_rank_fusion(rankings: Sequence[Sequence[int]], k: int = 60) -> list[tuple[int, float]]:
    """Fuse several ranked document-id lists (best first) into one, by RRF with constant `k`.

    Requirements, each of which is graded:
      * `score(d) = sum over rankings containing d of 1 / (k + rank_in_that_ranking(d))`, with
        rank counted from 1.
      * a document present in only SOME of the input rankings still appears in the result,
        with a score built only from the rankings it was actually in.
      * sorted by fused score descending, ties by document id ascending.

    Example:
        >>> reciprocal_rank_fusion([[1, 2, 3], [3, 1, 2]], k=60)[0][0]
        1
    """
    # YOUR CODE HERE
    raise NotImplementedError


CANDIDATE_N = 30   # how many top results each retriever contributes as fusion candidates
RRF_K = 60         # Cormack, Clarke & Büttcher's own default (see claims.yaml)


def hybrid_rank(query_tokens: Sequence[str], index: Index, dense_index: DenseIndex,
                k1: float = 1.2, b: float = 0.75, rrf_k: int = RRF_K,
                candidate_n: int = CANDIDATE_N) -> list[tuple[int, float]]:
    """Fuse BM25's and the dense retriever's top `candidate_n` results for `query_tokens`.

    Given, not graded: a thin composition of exercises 3, 6 and 7. When BM25 finds no shared
    term at all (its candidate list is empty — exercise 3's "removing the zeros"), the fused
    ranking is exactly the dense one; that graceful fallback is RRF's, not something special
    cased here.
    """
    bm25_ids = [doc_id for doc_id, _ in bm25_rank(query_tokens, index, k1, b)[:candidate_n]]
    dense_ids = [doc_id for doc_id, _ in dense_rank(query_tokens, dense_index)[:candidate_n]]
    return reciprocal_rank_fusion([bm25_ids, dense_ids], k=rrf_k)


# Public checks — run these as often as you like.
def _check_rrf() -> None:
    fused = reciprocal_rank_fusion([[10, 20, 30], [30, 10, 20]], k=60)
    fused_ids = [d for d, _ in fused]
    assert set(fused_ids) == {10, 20, 30}, f"expected all three ids present, got {fused_ids}"
    assert fused_ids[0] == 10, (
        f"doc 10 is rank 1 in list A and rank 2 in list B — the best combined rank of the "
        f"three — expected it first, got order {fused_ids}"
    )
    only_in_one = reciprocal_rank_fusion([[1, 2], [2, 3]], k=60)
    only_in_one_ids = {d for d, _ in only_in_one}
    assert only_in_one_ids == {1, 2, 3}, (
        f"a document present in only ONE list must still be fused in, got {only_in_one_ids}"
    )
    tied = [d for d, _ in reciprocal_rank_fusion([[7, 3], [3, 7]], k=60)]
    assert tied == [3, 7], (
        f"documents 3 and 7 fuse to exactly the same score, so the tie must break by "
        f"ascending document id — got {tied}"
    )
    small_k = dict(reciprocal_rank_fusion([[1, 2, 3]], k=1))
    large_k = dict(reciprocal_rank_fusion([[1, 2, 3]], k=1000))
    gap_small = small_k[1] - small_k[3]
    gap_large = large_k[1] - large_k[3]
    assert gap_small > gap_large > 0.0, (
        f"a SMALLER k should widen the gap between a top and a low rank ({gap_small:.5f}) "
        f"versus a LARGER k ({gap_large:.5f}), which flattens the list towards a tie"
    )
    print("exercise 7 looks right — reciprocal_rank_fusion merges partial candidate lists and "
          "its k constant controls how sharply top ranks are favoured")

In [ ]:
_try("exercise 7", _check_rrf)

In [ ]:
def _show_hybrid_on_both_query_kinds() -> None:
    docs, doc_meta = build_corpus()
    index = build_index(docs)
    dense_index = build_dense_index(index, DENSE_K)
    queries = build_queries(doc_meta)
    exact_q = next(q for q in queries if q.kind == "exact")
    para_q = next(q for q in queries if q.kind == "paraphrase")
    for q in (exact_q, para_q):
        ranked = hybrid_rank(q.tokens, index, dense_index)
        top_ids = [d for d, _ in ranked[:EVAL_K]]
        print(f"{q.kind:10s} query {q.tokens} -> target document in the top {EVAL_K}: "
              f"{any(d in q.relevant_ids for d in top_ids)}")

    print(f"\nthe target's position in the hybrid ranking, for each of the {len(queries)} "
          f"queries, at three values of RRF's k:")
    positions = {}
    for k in (1, RRF_K, 1000):
        positions[k] = []
        for q in queries:
            ranked = hybrid_rank(q.tokens, index, dense_index, rrf_k=k)
            positions[k].append(next((pos for pos, (d, _) in enumerate(ranked, start=1)
                                      if d in q.relevant_ids), None))
        print(f"  k={k:5d}  {positions[k]}")
    same = len({tuple(p) for p in positions.values()}) == 1
    print("the same at every k tried: " + ("yes" if same else "no"))


_try("hybrid on exact vs paraphrase", _show_hybrid_on_both_query_kinds,
     needs=("exercise 3", "exercise 6", "exercise 7"))

## 9. Exercise 8 — the instrument: recall@k, MRR and nDCG@k

`dcg_at_k` below (given) sums `relevance / log2(rank + 1)` over the top `k` of a ranking —
each document's relevance grade, discounted by how far down the list it sits. `ndcg_at_k`
divides that by the SAME sum computed on the best possible ordering, so a score of 1.0 always
means "as good as it could have been", whatever the query's own relevance grades happen to be.

The metrics read different judgements. Recall@k and MRR count only each query's target
(`relevant_ids`); nDCG@k reads the graded `relevance`, which also credits the subject's two
bridge documents at a lower grade — so nDCG can give credit to a ranking that recall and MRR
score as a miss.

<details><summary>💡 Hint — the approach, in words</summary>

`recall_at_k`: refuse an empty set of relevant ids before anything else. Then count how many
relevant ids sit among the first `k` ranked ids, and divide by how many relevant ids there
are — not by `k`. `reciprocal_rank`: walk the ranking from the top, counting from one, and
stop at the FIRST relevant id. `ndcg_at_k`: score the ranking you were given with
`dcg_at_k`; build the best possible ordering from `relevance` itself, highest grade first,
and score that with `dcg_at_k` at the same `k`; divide, unless the best score is zero.
</details>

In [ ]:
def dcg_at_k(ranked_ids: Sequence[int], relevance: Mapping[int, int], k: int) -> float:
    """Discounted cumulative gain of `ranked_ids` (best first) against `k`, using `relevance`.

    Given, not graded: `sum(relevance.get(doc_id, 0) / log2(rank + 1))` over the top `k` of
    `ranked_ids`, with `rank` counted from 1. A document absent from `relevance` contributes 0.
    """
    total = 0.0
    for rank, doc_id in enumerate(ranked_ids[:k], start=1):
        grade = relevance.get(doc_id, 0)
        if grade:
            total += grade / math.log2(rank + 1)
    return total


def recall_at_k(ranked_ids: Sequence[int], relevant_ids: Sequence[int] | frozenset[int],
                 k: int) -> float:
    """The fraction of `relevant_ids` that appear in the top `k` of `ranked_ids`.

    Requirements, each of which is graded:
      * `|top_k ∩ relevant_ids| / |relevant_ids|`, the textbook definition.
      * raises `ValueError` when `relevant_ids` is empty, rather than dividing 0 by 0.

    Example:
        >>> recall_at_k([5, 1, 2], {1, 9}, k=2)
        0.5
    """
    # YOUR CODE HERE
    raise NotImplementedError


def reciprocal_rank(ranked_ids: Sequence[int], relevant_ids: Sequence[int] | frozenset[int]) -> float:
    """1 / (rank of the first relevant document in `ranked_ids`), or 0.0 if none is present.

    Requirements, each of which is graded:
      * 1-based rank of the FIRST id in `ranked_ids` that is also in `relevant_ids`, reciprocated.
      * `0.0` if no id in `ranked_ids` is in `relevant_ids` — never an error.

    Example:
        >>> reciprocal_rank([5, 1, 2], {1, 9})
        0.5
        >>> reciprocal_rank([5, 6, 7], {1, 9})
        0.0
    """
    # YOUR CODE HERE
    raise NotImplementedError


def ndcg_at_k(ranked_ids: Sequence[int], relevance: Mapping[int, int], k: int) -> float:
    """Normalised DCG@k of `ranked_ids` against the graded judgements in `relevance`.

    Requirements, each of which is graded:
      * `dcg_at_k` of the ranking actually produced, divided by `dcg_at_k` of the IDEAL
        ordering — `relevance`'s own keys, sorted by grade descending.
      * `0.0`, not a division-by-zero error, when the ideal DCG (`idcg`) is itself `0.0`
        (an empty `relevance`, or every grade in it is 0).

    Example:
        >>> ndcg_at_k([1, 2], {1: 2, 2: 1}, k=2)
        1.0
        >>> ndcg_at_k([2, 1], {1: 2, 2: 1}, k=2) < 1.0
        True
    """
    # YOUR CODE HERE
    raise NotImplementedError


class EvalResult(NamedTuple):
    """Per-query metric values, in query order — the shape bootstrap_ci resamples over."""

    recall_at_k: list[float]
    reciprocal_rank: list[float]
    ndcg_at_k: list[float]


def evaluate_retriever(rank_fn: Callable[[Sequence[str]], list[tuple[int, float]]],
                        queries: Sequence[Query], k: int) -> EvalResult:
    """Run `rank_fn` over every query in `queries` and collect its per-query metrics.

    Given, not graded: a thin composition of this exercise's three functions. Returns
    PER-QUERY lists (not means) because exercise 9's bootstrap resamples over queries, and a
    mean thrown away here could never be recovered.
    """
    recalls, rrs, ndcgs = [], [], []
    for q in queries:
        ranked_ids = [doc_id for doc_id, _ in rank_fn(q.tokens)]
        recalls.append(recall_at_k(ranked_ids, q.relevant_ids, k))
        rrs.append(reciprocal_rank(ranked_ids, q.relevant_ids))
        ndcgs.append(ndcg_at_k(ranked_ids, q.relevance, k))
    return EvalResult(recalls, rrs, ndcgs)


# Public checks — run these as often as you like.
def _check_eval_metrics() -> None:
    assert recall_at_k([5, 1, 2], {1, 9}, k=2) == 0.5, "1 of 2 relevant ids in the top 2"
    assert recall_at_k([1, 9], {1, 9}, k=2) == 1.0, "both relevant ids in the top 2"
    assert recall_at_k([1, 5, 6], {1}, k=3) == 1.0, (
        "the one relevant id is in the top 3, so recall@3 is 1.0 — divide by how many ids "
        "are relevant, not by k (dividing by k gives precision@k)"
    )
    assert recall_at_k([5, 6, 1], {1}, k=2) == 0.0, (
        "the relevant id sits at rank 3, one place outside the top 2 — look at exactly k ids"
    )
    try:
        recall_at_k([1, 2], set(), k=2)
    except ValueError:
        pass
    else:
        raise AssertionError("recall_at_k with an EMPTY relevant_ids must raise ValueError")

    assert reciprocal_rank([5, 1, 2], {1, 9}) == 0.5, "first relevant id is at rank 2 -> 1/2"
    assert reciprocal_rank([1, 5, 2], {1, 9}) == 1.0, "first relevant id is at rank 1 -> 1/1"
    assert reciprocal_rank([5, 6, 7], {1, 9}) == 0.0, "no relevant id present -> 0.0"
    assert reciprocal_rank([7, 1, 9], {1, 9}) == 0.5, (
        "the FIRST relevant id (rank 2) decides it, not the last one (rank 3)"
    )

    perfect = ndcg_at_k([1, 2], {1: 2, 2: 1}, k=2)
    assert abs(perfect - 1.0) < 1e-9, f"the ideal ordering must score nDCG exactly 1.0, got {perfect}"
    swapped = ndcg_at_k([2, 1], {1: 2, 2: 1}, k=2)
    assert swapped < 1.0, "putting the lower-grade document first must score BELOW 1.0"
    assert ndcg_at_k([9, 9], {}, k=2) == 0.0, (
        "an empty relevance dict has an ideal DCG of 0 — must return 0.0, not raise ZeroDivisionError"
    )
    best_top_two = ndcg_at_k([1, 2], {1: 1, 2: 1, 3: 1}, k=2)
    assert abs(best_top_two - 1.0) < 1e-9, (
        f"two grade-1 documents in the top 2 is the best a top 2 can do, so nDCG@2 is 1.0, got "
        f"{best_top_two:.4f} — cut the IDEAL ordering at k too"
    )
    print("exercise 8 looks right — recall@k, reciprocal_rank and nDCG@k agree with hand-"
          "computed rankings, including the empty-relevance and empty-relevant-set edge cases")

In [ ]:
_try("exercise 8", _check_eval_metrics)

## 10. Exercise 9 — is it really better? `bootstrap_ci` and `paired_bootstrap_diff`

A mean over 10 or 20 queries is a point estimate, not a fact: swap in a different 10 queries
and it moves. A bootstrap resamples the queries THEMSELVES, with replacement, many times, and
reports the middle 95% of the resulting means — a range consistent with the queries you
measured, not a guess about queries you did not.

<details><summary>💡 Hint 1 — what to think about</summary>

Both functions need exactly ONE thing from numpy's random generator: a `(n_boot, n)` grid of
integer INDICES drawn with replacement from `range(n)`. Once you have that grid, every
resample's mean is one `.mean(axis=1)` call over the whole grid at once — no Python loop
over `n_boot`. For the paired version, the SAME grid of indices must be applied to both `a`
and `b` before subtracting, or a query's own difficulty adds noise instead of cancelling out.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Make a generator from the seed inside the function. Draw the whole grid of resample indices
in one call, look the values up through it, and average each resample. The interval's ends
are two percentiles of those averages: alpha is split evenly between the two tails, and
numpy's percentile function counts in percent, not as a fraction. Return the plain mean of
the original values with them. The paired version draws ONE grid, looks up both `a` and `b`
through it, and averages the per-query differences, `b` minus `a`.
</details>

In [ ]:
def bootstrap_ci(values: Sequence[float], n_boot: int = N_BOOT, seed: int = 0,
                  alpha: float = 0.05) -> tuple[float, float, float]:
    """A percentile bootstrap confidence interval for the mean of `values`.

    Returns `(mean, lower, upper)`: `mean` is the plain mean of `values`; `(lower, upper)` are
    the `100*alpha/2` and `100*(1-alpha/2)` percentiles of `n_boot` resampled means.

    Requirements, each of which is graded:
      * `n_boot` resamples, each of `len(values)` indices drawn WITH replacement, from
        `np.random.default_rng(seed)` — the SAME seed must give the SAME interval every time.
      * the interval's ends are the `100*alpha/2` and `100*(1-alpha/2)` percentiles, so the
        default `alpha = 0.05` gives a 95% interval.

    Aim for one `(n_boot, n)` array of resample indices and one `.mean(axis=1)` call; a Python
    loop over `n_boot` gives the same kind of interval, just more slowly.

    Example:
        >>> lo_hi = bootstrap_ci([1.0, 1.0, 1.0], n_boot=200, seed=1)[1:]
        >>> lo_hi == (1.0, 1.0)
        True
    """
    # YOUR CODE HERE
    raise NotImplementedError


def paired_bootstrap_diff(a: Sequence[float], b: Sequence[float], n_boot: int = N_BOOT,
                           seed: int = 0, alpha: float = 0.05) -> tuple[float, float, float]:
    """A percentile bootstrap confidence interval for the mean of `b[i] - a[i]`, paired by query.

    `a` and `b` must be the SAME metric from two retrievers over the SAME queries in the same
    order — `evaluate_retriever`'s output for each.

    Requirements, each of which is graded:
      * each resample draws ONE set of query indices and applies it to BOTH `a` and `b` before
        differencing — never two independent resamples of `a` and of `b`.
      * otherwise identical to `bootstrap_ci`: `n_boot` resamples, `np.random.default_rng(seed)`,
        percentiles at `100*alpha/2` and `100*(1-alpha/2)`.

    Example:
        >>> mean_diff, lo, hi = paired_bootstrap_diff([0.0, 0.0], [1.0, 1.0], n_boot=200, seed=1)
        >>> mean_diff, lo, hi
        (1.0, 1.0, 1.0)
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_bootstrap() -> None:
    mean, lo, hi = bootstrap_ci([1.0, 1.0, 1.0, 1.0], n_boot=500, seed=3)
    assert mean == 1.0 and lo == 1.0 and hi == 1.0, (
        "with every value identical, the interval must collapse to exactly that value"
    )
    values = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 0.5, 0.3, 0.7, 0.9]
    mean1, lo1, hi1 = bootstrap_ci(values, n_boot=500, seed=1)
    mean2, lo2, hi2 = bootstrap_ci(values, n_boot=500, seed=1)
    assert (mean1, lo1, hi1) == (mean2, lo2, hi2), (
        "the SAME seed must give the SAME interval every time — draw from "
        "np.random.default_rng(seed), not the global numpy random state"
    )
    assert lo1 <= mean1 <= hi1, "the mean must sit inside its own interval"

    mean_diff, lo, hi = paired_bootstrap_diff([0.0, 0.0, 0.0], [1.0, 1.0, 1.0], n_boot=500, seed=2)
    assert mean_diff == 1.0 and lo == 1.0 and hi == 1.0, (
        "b uniformly 1.0 higher than a, paired, must give a degenerate interval at exactly 1.0"
    )
    a = [1.0, 0.0, 1.0, 0.0, 1.0]
    b = [0.0, 1.0, 0.0, 1.0, 0.0]
    _, lo_diff, hi_diff = paired_bootstrap_diff(a, b, n_boot=500, seed=4)
    assert lo_diff < 0.0 < hi_diff, (
        "a and b disagree on every query in a way that averages to zero either direction — "
        "the interval should straddle 0, not exclude it"
    )
    spread = np.linspace(0.0, 1.0, 50)
    half_width = statistics.NormalDist().inv_cdf(0.975) * float(spread.std()) / math.sqrt(50)
    _, lo95, hi95 = bootstrap_ci(spread, n_boot=20000, seed=6)
    assert abs(lo95 - (0.5 - half_width)) < 0.006 and abs(hi95 - (0.5 + half_width)) < 0.006, (
        f"a 95% interval for this mean should run from about {0.5 - half_width:.3f} to "
        f"{0.5 + half_width:.3f}, got [{lo95:.3f}, {hi95:.3f}] — take the 100*alpha/2 and "
        "100*(1-alpha/2) percentiles: alpha is split between the two tails"
    )
    print("exercise 9 looks right — bootstrap_ci and paired_bootstrap_diff are reproducible "
          "under a fixed seed and collapse correctly on degenerate inputs")

In [ ]:
_try("exercise 9", _check_bootstrap)

## 11. The scorecard: three retrievers, two slices, and the interval that decides it

Every number below is computed by the functions above, run once, here. Read the exact and
paraphrase rows first — that is the planted contrast this corpus exists to show. Then read
the paired differences at the foot, slice by slice: an interval that excludes 0 is a real
difference; an interval that includes 0 is one these queries cannot tell from noise, however
the two means compare. The last line pools the two pure retrievers over both slices —
compare it with the slice-by-slice lines above it before you believe it.

In [ ]:
def run_full_scorecard() -> None:
    docs, doc_meta = build_corpus()
    index = build_index(docs)
    dense_index = build_dense_index(index, DENSE_K)
    queries = build_queries(doc_meta)
    exact_qs = [q for q in queries if q.kind == "exact"]
    para_qs = [q for q in queries if q.kind == "paraphrase"]

    retrievers: dict[str, Callable[[Sequence[str]], list[tuple[int, float]]]] = {
        "bm25": lambda toks: bm25_rank(toks, index),
        "dense": lambda toks: dense_rank(toks, dense_index),
        "hybrid": lambda toks: hybrid_rank(toks, index, dense_index),
    }
    print(f"{index.n_docs} documents, {len(exact_qs)} exact queries, {len(para_qs)} "
          f"paraphrase queries, recall/nDCG cut off at k={EVAL_K}\n")
    for slice_name, qs in (("exact", exact_qs), ("paraphrase", para_qs), ("all", queries)):
        print(f"-- {slice_name} (n={len(qs)}) --")
        for name, rank_fn in retrievers.items():
            ev = evaluate_retriever(rank_fn, qs, EVAL_K)
            r_mean, r_lo, r_hi = bootstrap_ci(ev.recall_at_k, seed=SEED)
            mrr_mean, mrr_lo, mrr_hi = bootstrap_ci(ev.reciprocal_rank, seed=SEED)
            nd_mean, nd_lo, nd_hi = bootstrap_ci(ev.ndcg_at_k, seed=SEED)
            print(f"  {name:7s} recall@{EVAL_K}={r_mean:.2f} [{r_lo:.2f},{r_hi:.2f}]   "
                  f"mrr={mrr_mean:.2f} [{mrr_lo:.2f},{mrr_hi:.2f}]   "
                  f"ndcg@{EVAL_K}={nd_mean:.2f} [{nd_lo:.2f},{nd_hi:.2f}]")
        print()

    def verdict(lo: float, hi: float) -> str:
        return ("real: the interval excludes 0" if (lo > 0 or hi < 0)
                else "not shown: the interval includes 0")

    print("-- paired differences in MRR, slice by slice --")
    real = []
    for slice_name, qs in (("exact", exact_qs), ("paraphrase", para_qs), ("all", queries)):
        results = {name: evaluate_retriever(fn, qs, EVAL_K) for name, fn in retrievers.items()}
        for other in ("bm25", "dense"):
            diff, lo, hi = paired_bootstrap_diff(results[other].reciprocal_rank,
                                                  results["hybrid"].reciprocal_rank, seed=SEED)
            real.append(lo > 0 or hi < 0)
            print(f"  {slice_name:10s} hybrid - {other:5s}  {diff:+.3f} [{lo:+.3f},{hi:+.3f}]  "
                  f"->  {verdict(lo, hi)}")
    diff, lo, hi = paired_bootstrap_diff(results["bm25"].reciprocal_rank,
                                          results["dense"].reciprocal_rank, seed=SEED)
    print(f"  {'all':10s} dense  - bm25   {diff:+.3f} [{lo:+.3f},{hi:+.3f}]  ->  {verdict(lo, hi)}")
    print(f"\nhybrid's advantage is real in {sum(real)} of the {len(real)} hybrid comparisons "
          f"and not shown in {len(real) - sum(real)}.")
    if sum(real) < len(real):
        print("Where an interval includes 0, these queries cannot tell the two retrievers apart,")
        print("whatever their two means say. A table of means alone would not have told you.")


_try("full scorecard", run_full_scorecard,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 5",
            "exercise 6", "exercise 7", "exercise 8", "exercise 9"))

## 12. Common mistakes

- **Deduplicating query terms in `bm25_score`.** Robertson and Zaragoza treat a repeated
  query term as a separate occurrence (`qtf`), not a no-op; skipping repeats under-scores
  any query that repeats a word.
- **Scoring a document that shares nothing with the query.** BM25's own zero-contribution
  terms must be REMOVED from the ranking, not left in at score 0.0 and sorted to the bottom —
  a real inverted index never looks such a document up at all.
- **Clamping `bm25_idf` at zero.** The primary-literature formula is allowed to go negative
  for a term that sits in more than half the documents; flooring it at 0 is a different formula
  from the one this lesson implements, and it erases the edge case exercise 2 measures.
- **Building `tfidf_matrix` with `bm25_idf`.** The two idf functions exist because a vector
  space has no "removing the zeros" step to protect it — a large negative BM25 weight on a
  near-universal term would swamp a short, otherwise on-topic row.
- **Normalising a query with a different rule than the documents.** Cosine similarity is only
  a plain dot product when BOTH sides are unit length; embed and normalise a query exactly
  the way `build_dense_index` embeds and normalises a document, via the same `Vt_k`.
- **Dividing by a zero norm.** An out-of-vocabulary query, or a document with no tokens at
  all, has an all-zero raw vector; guard the division, or the result is `nan`, silently
  poisoning every downstream score it touches.
- **Treating a missing document as score 0 in `reciprocal_rank_fusion`.** A document absent
  from one ranked list contributes nothing from THAT list, not a zero-ranked contribution —
  the two are different numbers.
- **Confusing `dcg_at_k`'s "ideal" ordering with sorting the RANKING you were given.** The
  ideal ordering for nDCG's denominator sorts `relevance`'s own keys by grade, independent of
  whatever order any retriever actually produced.
- **Reading a point estimate as a verdict.** Two means 0.05 apart, from 10 or 20 queries, are
  not "the better one" until a bootstrap interval says the gap survives resampling. The
  paired intervals at the foot of the scorecard are where you check.
- **Treating RRF's `k` as a cosmetic scale.** With one input list it only rescales. With two
  or more it sets how much one list's single top rank counts against agreement between the
  lists (Cormack, Clarke and Büttcher describe it as mitigating the impact of high rankings
  by outlier systems), so changing it can reorder the fused list.

That last mistake, on two small lists, measured rather than asserted:

In [ ]:
def _show_rrf_k_in_practice() -> None:
    # Document 1 is rank 1 in list A and rank 30 in list B; document 2 is rank 13 in both.
    # Every other id is filler that appears in one list only.
    list_a = [1] + list(range(100, 111)) + [2]
    list_b = list(range(200, 212)) + [2] + list(range(300, 316)) + [1]
    assert list_a.index(2) + 1 == 13 and list_b.index(2) + 1 == 13 and list_b.index(1) + 1 == 30
    first = {}
    for k in (1, 60, 120, 1000):
        fused = reciprocal_rank_fusion([list_a, list_b], k=k)
        score = dict(fused)
        first[k] = next(d for d, _ in fused if d in (1, 2))
        print(f"k={k:5d}  doc 1 (ranks 1 and 30) {score[1]:.6f}   doc 2 (ranks 13 and 13) "
              f"{score[2]:.6f}   -> doc {first[k]} first")
    if len(set(first.values())) > 1:
        print("\nthe same two lists, fused in a different order: a small k rewards one list's top")
        print("rank, a large k rewards agreement between the lists.")
    else:
        print("\nthe order did not change at these values of k.")


_try("rrf k in practice", _show_rrf_k_in_practice, needs=("exercise 7",))

## 13. Self-check

1. Your `bm25_idf` is handed a term that appears in every one of the corpus's documents.
   What should it return?
   - (a) exactly `0.0` — matching such a term is neither evidence for nor against a document
   - (b) a finite, NEGATIVE number
   - (c) `-inf`, because the numerator `N - n_t` is `0`

2. A paraphrase query's tokens never once appear, literally, in its target document. On the
   corpus in this lesson, `bm25_rank` on that query:
   - (a) drops the target document from the ranking entirely, however good a match it is in
     meaning
   - (b) still ranks the target document, at a low but non-zero score
   - (c) raises an exception, because none of the query's terms are in `index.df`

3. Every sibling document carries the same three topic words as its target, and a
   different case code. For an exact query — the target's case code alone — why can BM25
   put the target first when the dense retriever may not?
   - (a) the truncated SVD is buggy — a correct one would always match BM25 on these queries
   - (b) the case code has a higher idf than the shared words, which should hurt BM25
   - (c) in the reduced space the code has no dimension of its own, so cosine similarity sees
     the target and its siblings as near-identical; BM25 matches the code as a literal term

4. Two retrievers' rankings are fused with RRF. Raising `k` from 60 to 120, with the same
   input rankings:
   - (a) can never change the fused ORDER, only shrink the gaps, since `k` is added to
     every rank alike
   - (b) can change the fused order: a larger `k` weakens one list's single top rank
     relative to agreement between the lists
   - (c) changes which documents appear in the fused list

5. Your scorecard's point estimates show retriever A with a higher mean MRR than retriever B
   over the combined 20-query set, but the paired bootstrap interval for "B minus A" straddles
   zero. The correct conclusion is:
   - (a) A is better — the mean says so, and that is what a mean is for
   - (b) this many queries cannot distinguish the two; report the interval, not just the mean
   - (c) the bootstrap must be broken, since it disagrees with the mean

Answers come with this lesson's worked solution when you enrol on Synapsa.

## What you built

`Index`, `DenseIndex`, `Query` and `EvalResult` are the record shapes; `bm25_rank`,
`dense_rank` and `hybrid_rank` are the three retrievers; `evaluate_retriever`,
`bootstrap_ci` and `paired_bootstrap_diff` are the instrument. Keep them together: a
retriever is only as trustworthy as the instrument that measured it, and the instrument is
only as trustworthy as the intervals it reports.

In [ ]:
# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    _ALL_CHECKS = (  # (exercise, its check, the exercises that check relies on)
        ("exercise 1", _check_index, ()),
        ("exercise 2", _check_bm25_core, ()),
        ("exercise 3", _check_bm25_rank, ()),
        ("exercise 4", _check_tfidf_matrix, ()),
        ("exercise 5", _check_truncated_svd, ()),
        ("exercise 6", _check_dense_retriever, ("exercise 4", "exercise 5")),
        ("exercise 7", _check_rrf, ()),
        ("exercise 8", _check_eval_metrics, ()),
        ("exercise 9", _check_bootstrap, ()),
    )
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in _ALL_CHECKS:
            _try(_name, _check, needs=_needs)
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends the run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.